# Train ICPR U-Net on Kaggle — 4-fold CV + fixed-test evaluation

Trains the **image-only ICPR U-Net baseline** under the same protocol as TransUNet (4-fold CV + fixed-test eval). Like TransUNet, it needs only `data/splits` — no BB priors.

## Before you run
1. **Accelerator:** Settings → **GPU T4 x2** or **GPU P100**.
2. **Internet:** Settings → **On** (for `git`, `pip`).
3. **Data:** right panel → *Add Input* → your **pbl4-splits** dataset.

Then **Run All**, top to bottom. Re-running is safe: each cell re-establishes
its own state, and training skips folds that already finished.

The `icpr_unet` preset uses batch 1, no mixed precision, 60 epochs, and no early stopping or process-restart loop — the simplest image-only baseline.

## To download results afterwards
Files in `/kaggle/working` only appear in the **Output tab** after you save a
version. Use **Save Version → Quick Save** (with *Save output* on) — it
snapshots the current outputs **without** re-running. Or grab the zips directly
from the link printed by the last cell. (Do **not** use *Save & Run All* just to
download — it retrains from scratch.)

## 1. Configure

In [ ]:
REPO_URL = "https://github.com/Huay0804/PBL4.git"  # private? use https://<TOKEN>@github.com/Huay0804/PBL4.git
REPO_DIR = "/kaggle/working/PBL4"
MODEL    = "icpr_unet"   # wired for this notebook; don't change

## 2. Get the latest code and use Kaggle's preinstalled stack
Clones if missing, then **always fast-forwards to `origin/main`** so you never
run stale scripts. `git reset --hard` only touches tracked files — your `runs/`
and `data/` outputs are left alone. Uses Kaggle's preinstalled TensorFlow +
Keras 3 (we deliberately do **not** pip-upgrade numpy/TF: scipy and
scikit-image on Kaggle are built against the preinstalled numpy ABI, so an
upgrade leaves them half-broken and unrecoverable in-session).

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=True)
print("code:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

try:
    import tensorflow as tf
    import keras
except Exception as e:
    raise SystemExit(
        f"Environment broken (TF import failed): {e}\n\n"
        "Stop session (top-right) and start a fresh one — a previous run may "
        "have pip-upgraded numpy/TF and left a half-broken install."
    )
print(f"TF {tf.__version__} | Keras {keras.__version__}")
print("GPUs:", tf.config.list_physical_devices("GPU") or "NONE — Settings → Accelerator")
assert keras.__version__.startswith("3."), f"Need Keras 3.x; have {keras.__version__}"

## 3. Wire up data and sanity-check the environment
`ensure_splits()` symlinks `data/splits` → the mounted splits dataset (auto-
detected under `/kaggle/input` by globbing for `class_map.txt`). Helpers
are redefined in every data-touching cell so any cell can be run on its own
after a kernel restart.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _splits_root():
    """Return the splits/ directory inside the mounted pbl4-splits dataset."""
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input — "
                                "add the pbl4-splits dataset (right panel -> Add Input).")
    return os.path.dirname(hits[0])

def ensure_splits():
    """Idempotently link data/splits -> the mounted splits dataset."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    sp = _splits_root()
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)

ensure_splits()

import sys, os
sys.path.insert(0, os.path.abspath("src"))
import tensorflow as tf, keras
print(f"TF {tf.__version__} | Keras {keras.__version__}")
print("GPUs:", tf.config.list_physical_devices("GPU") or "NONE — confirm Settings → Accelerator")

from segmentation_models import ICPRUnet
_m = ICPRUnet(input_shape=(512, 1024, 3), classes=33, activation="softmax")
print(f"icpr_unet params: {_m.count_params():,}")
del _m

## 4. Train all 4 CV folds
Same command as local — the preset drives batch size, mixed precision (if enabled), early-stopping (where configured), and the process-restart loop. **Folds with `best.keras` already saved are skipped**, so
interrupted runs resume cheaply. `PBL4_GPU_DISPLAY_RESERVE_MB=0` uses the full
headless cloud GPU.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _splits_root():
    """Return the splits/ directory inside the mounted pbl4-splits dataset."""
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input — "
                                "add the pbl4-splits dataset (right panel -> Add Input).")
    return os.path.dirname(hits[0])

def ensure_splits():
    """Idempotently link data/splits -> the mounted splits dataset."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    sp = _splits_root()
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)

ensure_splits()

import os
os.environ["PBL4_GPU_DISPLAY_RESERVE_MB"] = "0"

for k in range(4):
    fold_root = f"runs/cv/fold_{k}/icpr_unet"
    already_done = False
    if os.path.isdir(fold_root):
        for r, _, files in os.walk(fold_root):
            if "best.keras" in files:
                already_done = True
                break
    if already_done:
        print(f"skip icpr_unet fold {k} (best.keras already present under {fold_root})")
        continue
    print(f"\n========== TRAIN icpr_unet fold {k} ==========", flush=True)
    rc = os.system(f"python -u scripts/train_segmentation_cv.py --model icpr_unet --fold {k}")
    if rc != 0:
        raise SystemExit(f"icpr_unet fold {k} training failed (exit {rc}).")
print("\nAll folds present.")

## 5. Evaluate each fold on the fixed test set
Writes `test_summary.json`, `test_metrics.json`, `per_class_metrics_test.json`,
`per_position_metrics_test.json`, and `per_tooth_type_metrics_test.json` next to
each fold's checkpoint. Format is unchanged from the other segmentation models,
so the per-fold numbers are directly comparable.

In [ ]:
import os, glob, shutil, sys, subprocess
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.abspath("scripts"))

def _splits_root():
    """Return the splits/ directory inside the mounted pbl4-splits dataset."""
    hits = glob.glob("/kaggle/input/**/class_map.txt", recursive=True)
    if not hits:
        raise FileNotFoundError("No class_map.txt under /kaggle/input — "
                                "add the pbl4-splits dataset (right panel -> Add Input).")
    return os.path.dirname(hits[0])

def ensure_splits():
    """Idempotently link data/splits -> the mounted splits dataset."""
    os.chdir(REPO_DIR)
    if os.path.exists("data/splits/class_map.txt"):
        return
    sp = _splits_root()
    os.makedirs("data", exist_ok=True)
    link = "data/splits"
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    elif os.path.exists(link): os.remove(link)
    os.symlink(sp, link)
    print("linked data/splits ->", sp)

ensure_splits()

import os
for k in range(4):
    print(f"\n========== EVAL icpr_unet fold {k} ==========", flush=True)
    rc = os.system(f"python -u scripts/evaluate_final.py --model icpr_unet --cv-fold {k}")
    if rc != 0:
        raise SystemExit(f"icpr_unet fold {k} evaluation failed (exit {rc}).")
print("\nAll evaluations done.")

## 6. Results
Cross-validation aggregate (validation folds) from the CV summary, plus the
per-fold test-set summaries.

In [ ]:
import glob, json, os
os.chdir(REPO_DIR)

cv = "runs/cv/icpr_unet_cv_summary.json"
if os.path.exists(cv):
    agg = (json.load(open(cv)) or {}).get("aggregate", {})
    print("=== CV aggregate (validation) ===")
    print(json.dumps(agg, indent=2))
else:
    print(f"(no {cv} yet)")

print("\n=== Per-fold test-set summaries ===")
for p in sorted(glob.glob("runs/cv/fold_*/icpr_unet/**/test_summary.json", recursive=True)):
    print(p)
    print(json.dumps(json.load(open(p)), indent=2))

## 7. Package results for download
Writes **two** zips to `/kaggle/working`:
- `icpr_unet_models.zip` — only the `best.keras` checkpoints (the heavy files).
- `icpr_unet_results.zip` — all metrics, CV summary and metadata (no `.keras`, no
  TensorBoard logs, no `BackupAndRestore` snapshots).

**Downloading on Kaggle:** the editor's **right panel → Output → `/kaggle/working`**
(download icon next to each zip) is the reliable path. Or **Save Version →
Quick Save** (with *Save output* on) and grab from the saved version's **Output
tab**. (Do **not** use *Save & Run All* just to download — it retrains.)

In [ ]:
import os, glob, subprocess
os.chdir(REPO_DIR)

models_zip  = "/kaggle/working/icpr_unet_models.zip"
results_zip = "/kaggle/working/icpr_unet_results.zip"
for z in (models_zip, results_zip):
    if os.path.exists(z):
        os.remove(z)

best_files = sorted({
    p for p in glob.glob("runs/cv/fold_*/icpr_unet/**/best.keras", recursive=True)
    if "/.training_backup/" not in p
})
if not best_files:
    raise SystemExit("No best.keras under runs/cv/fold_*/icpr_unet/ — run training (and eval) first.")

subprocess.run(["zip", "-q", models_zip, *best_files], check=True)
subprocess.run([
    "zip", "-r", "-q", results_zip, "runs/cv",
    "-x", "*.keras", "*/logs/*", "*/.training_backup/*",
], check=True)

print(f"models  : {models_zip}  ({os.path.getsize(models_zip)/1e6:.1f} MB, {len(best_files)} best.keras)")
print(f"results : {results_zip} ({os.path.getsize(results_zip)/1e6:.1f} MB)")
print("\nDownload:")
print("  - right panel -> Output -> /kaggle/working -> download icon next to each zip")
print("  - or Save Version -> Quick Save (Save output) -> version Output tab -> Download")